# Blind Image Restoration via Conditional Diffusion

Single diffusion model that restores images degraded by an **unknown mix** of Gaussian blur, noise, and JPEG artifacts — trained on DIV2K, evaluated per-degradation against classical baselines.

**Runtime:** enable GPU (T4/P100) and Internet. Training ~4–6 h for 20k steps; reduce `STEPS` for a quick run.


## 1. Setup

In [ ]:
import os, sys
REPO = "/kaggle/working/blind-diffusion-restoration"
if not os.path.exists(REPO):
    !git clone https://github.com/DevZyan/blind-diffusion-restoration.git {REPO}
sys.path.append(os.path.join(REPO, "src"))
!pip install -q diffusers lpips
print("ready")

In [ ]:
import torch, warnings
warnings.filterwarnings("ignore")
print("CUDA:", torch.cuda.is_available())

## 2. Data

Add the **DIV2K** dataset to the notebook and update the paths below.

In [ ]:
TRAIN_DIR = "/kaggle/input/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR"
VAL_DIR   = "/kaggle/input/div2k-dataset/DIV2K_valid_HR/DIV2K_valid_HR"
OUT       = "/kaggle/working/ckpt"
import glob
print("train imgs:", len(glob.glob(TRAIN_DIR + "/*.png")))
print("val imgs:  ", len(glob.glob(VAL_DIR + "/*.png")))

## 3. Inspect the blind degradation pipeline

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from degradations import BlindDegrader
img = np.asarray(Image.open(sorted(glob.glob(VAL_DIR+"/*.png"))[0]).convert("RGB"), np.float32)/255
img = img[:256,:256]
deg = BlindDegrader(seed=0)
fig, ax = plt.subplots(1,5,figsize=(18,4))
ax[0].imshow(img); ax[0].set_title("clean"); ax[0].axis("off")
for i in range(1,5):
    d,meta = deg.degrade(img)
    ax[i].imshow(d); ax[i].axis("off"); ax[i].set_title(", ".join(meta["ops"]), fontsize=9)
plt.show()

## 4. Train

Set `PRETRAINED` to a diffusers DDPM id to fine-tune from a prior, or leave empty to train the compact U-Net from scratch.

In [ ]:
STEPS = 20000
PRETRAINED = ""
cmd = f'python {REPO}/src/train.py --data "{TRAIN_DIR}" --out "{OUT}" --patch 128 --batch 8 --steps {STEPS} --lr 2e-4 --timesteps 1000 --schedule cosine --ckpt_every 2000 --sample_every 2000 --workers 2'
if PRETRAINED: cmd += f' --pretrained {PRETRAINED}'
print(cmd)
!{cmd}

### Training previews
`degraded | restored | ground-truth`

In [ ]:
from IPython.display import Image as IPImage, display
samples = sorted(glob.glob(OUT + "/sample_*.png"))
if samples: display(IPImage(filename=samples[-1]))

## 5. Evaluate — per-degradation table vs baselines

In [ ]:
!python {REPO}/src/evaluate.py --data "{VAL_DIR}" --ckpt "{OUT}/latest.pt" --out /kaggle/working/results --limit 50 --size 256 --ddim_steps 50 --lpips --baselines identity,nlm,wiener,classical_blind

In [ ]:
print(open("/kaggle/working/results/results.md").read())

## 6. Qualitative comparison figure

In [ ]:
!python {REPO}/src/visualize.py --data "{VAL_DIR}" --ckpt "{OUT}/latest.pt" --out /kaggle/working/comparison.png --n 3
display(IPImage(filename="/kaggle/working/comparison.png"))

## 7. Notes
- **DDIM** (50 steps) for eval; full **DDPM** (1000 steps) via `diffusion.sample()` for max quality.
- **EMA weights** used at eval — they stabilize samples.
- To improve: train longer, add an image-space L1 on predicted x0, or fine-tune from a pretrained DDPM prior.
